In [1]:
# ============================================================
# NOTEBOOK 02 — CHUNKING
# ============================================================
# WHAT IS CHUNKING?
# After ingestion we have the full document in memory.
# But each page can be very long — too long for an LLM
# to process efficiently.
#
# Chunking = splitting the text into smaller, manageable pieces
#
# Two key settings:
#   chunk_size    : how many characters per chunk
#   chunk_overlap : how many characters to repeat between
#                   neighbouring chunks (to preserve context)
#
# Example with chunk_size=20, chunk_overlap=5:
#   Text    : "The cat sat on the mat"
#   Chunk 1 : "The cat sat on the"
#   Chunk 2 : "on the mat"
#              ^^^^^^ this overlap keeps context connected
# ============================================================

print("Starting chunking notebook...")

Starting chunking notebook...


In [2]:
# PyPDFLoader to reload our document
from langchain_community.document_loaders import PyPDFLoader

# RecursiveCharacterTextSplitter is the most common chunking tool
# It tries to split on paragraphs first, then sentences, then words
# This keeps chunks as meaningful as possible
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("Libraries imported!")

Libraries imported!


In [3]:
# ============================================================
# STEP 1: RELOAD THE DOCUMENT FROM INGESTION
# ============================================================

# We reload the same PDF we used in notebook 01
pdf_path = "../data/documents/attention_is_all_you_need.pdf"

# Load the PDF — same as before
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Document reloaded: {len(documents)} pages")

Document reloaded: 15 pages


In [4]:
# ============================================================
# STEP 2: CREATE THE TEXT SPLITTER
# ============================================================

# RecursiveCharacterTextSplitter splits text intelligently
# It tries these separators in order: \n\n, \n, " ", ""
# This means it prefers to split on paragraph breaks first

text_splitter = RecursiveCharacterTextSplitter(

    # Each chunk will be at most 500 characters long
    # 500 is a good balance — not too big, not too small
    chunk_size=500,

    # 50 characters will overlap between neighbouring chunks
    # This ensures we don't lose context at chunk boundaries
    chunk_overlap=50,

    # We measure length by number of characters
    length_function=len,

    # If True, adds chunk metadata automatically
    add_start_index=True
)

print("Text splitter created!")
print(f"Chunk size    : 500 characters")
print(f"Chunk overlap : 50 characters")

Text splitter created!
Chunk size    : 500 characters
Chunk overlap : 50 characters


In [5]:
# ============================================================
# STEP 3: SPLIT THE DOCUMENTS INTO CHUNKS
# ============================================================

# split_documents takes our list of page Documents
# and returns a new list of smaller chunk Documents
# Each chunk is still a Document object with content + metadata
chunks = text_splitter.split_documents(documents)

print(f"Original pages : {len(documents)}")
print(f"Total chunks   : {len(chunks)}")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

Original pages : 15
Total chunks   : 94
Average chunk size: 429 characters


In [6]:
# ============================================================
# STEP 4: EXPLORE WHAT A CHUNK LOOKS LIKE
# ============================================================

# Let's look at the first 3 chunks in detail
for i, chunk in enumerate(chunks[:3]):

    print(f"\n{'='*50}")
    print(f"CHUNK {i+1}")
    print(f"{'='*50}")

    # The metadata now includes the original page number
    # AND the start_index (where in the page this chunk began)
    print(f"Metadata : {chunk.metadata}")

    # The actual text content of this chunk
    print(f"Length   : {len(chunk.page_content)} characters")
    print(f"Content  : {chunk.page_content}")


CHUNK 1
Metadata : {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/documents/attention_is_all_you_need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'start_index': 0}
Length   : 492 characters
Content  : Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan

In [7]:
# ============================================================
# STEP 5: SEE THE OVERLAP IN ACTION
# ============================================================
# Let's prove that overlap is working by comparing
# the end of chunk 1 with the start of chunk 2

print("=== END OF CHUNK 1 ===")
# Show last 100 characters of chunk 1
print(repr(chunks[0].page_content[-100:]))

print("\n=== START OF CHUNK 2 ===")
# Show first 100 characters of chunk 2
print(repr(chunks[1].page_content[:100]))

print("\nNotice the repeated text — that is the overlap working!")

=== END OF CHUNK 1 ===
'Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu'

=== START OF CHUNK 2 ===
'University of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia'

Notice the repeated text — that is the overlap working!


In [8]:
# ============================================================
# STEP 6: EXPERIMENT — WHAT HAPPENS WITH DIFFERENT SIZES?
# ============================================================
# This is important to understand for interviews!
# Larger chunks = more context but slower and costlier
# Smaller chunks = faster but might miss context

for size in [200, 500, 1000]:

    # Create a splitter with this chunk size
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=50
    )

    # Split and count
    test_chunks = splitter.split_documents(documents)

    print(f"chunk_size={size:4d} → {len(test_chunks):3d} chunks")

chunk_size= 200 → 265 chunks
chunk_size= 500 →  94 chunks
chunk_size=1000 →  48 chunks


In [9]:
# ============================================================
# SUMMARY — WHAT DID WE LEARN?
# ============================================================

print("=== CHUNKING SUMMARY ===")
print(f"Total chunks created : {len(chunks)}")
print(f"Chunk size used      : 500 characters")
print(f"Chunk overlap used   : 50 characters")
print(f"Each chunk type      : {type(chunks[0])}")

print("""
WHAT HAPPENS NEXT?
------------------
Now we have hundreds of small text chunks.
The next step is VECTORIZATION.

Vectorization means converting each chunk into a
list of numbers called an "embedding".

These numbers capture the MEANING of the text so
we can later find the most relevant chunks for
any question mathematically.

See notebook 03_vectorization.ipynb for the next step!
""")

=== CHUNKING SUMMARY ===
Total chunks created : 94
Chunk size used      : 500 characters
Chunk overlap used   : 50 characters
Each chunk type      : <class 'langchain_core.documents.base.Document'>

WHAT HAPPENS NEXT?
------------------
Now we have hundreds of small text chunks.
The next step is VECTORIZATION.

Vectorization means converting each chunk into a
list of numbers called an "embedding".

These numbers capture the MEANING of the text so
we can later find the most relevant chunks for
any question mathematically.

See notebook 03_vectorization.ipynb for the next step!

